# DAMGF-Net: reproduce the controlled experiments in Google Colab

This notebook reproduces Sections 4.7 and 5.6-5.9 of the paper and writes the released data files
(splits, de-duplication lists, pairing indices).

**Before you start:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells in order (Shift + Enter).

Total time: about 1-2 hours on a T4. All steps are resumable: if Colab disconnects, run the cells again
and finished steps are skipped (results are kept in Google Drive).

In [ ]:
# 1. Mount Google Drive (results are stored there so that the run survives disconnects)
from google.colab import drive
drive.mount('/content/drive')
RUN = '/content/drive/MyDrive/DAMGF-Net_run'
import os; os.makedirs(RUN, exist_ok=True)
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to GPU')

In [ ]:
# 2. Get the code
# Clones the public repository (if cloning fails, upload a zip of the repository when the file picker opens).
REPO_URL = 'https://github.com/dmounikaresearch-cmyk/DAMGF-Net.git'
import os, subprocess
if not os.path.isdir('/content/DAMGF-Net'):
    r = subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/DAMGF-Net'])
    if r.returncode != 0:
        from google.colab import files
        print('Clone failed - choose a zip of the repository')
        up = files.upload()
        os.system(f'unzip -q "{list(up)[0]}" -d /content/')
%cd /content/DAMGF-Net
!pip -q install timm transformers imagehash fvcore tabulate kaggle

In [ ]:
# 3. Download the datasets from Kaggle (choose your kaggle.json when asked)
import os, shutil
if not os.path.exists('/root/.kaggle/kaggle.json'):
    from google.colab import files
    print('Choose your kaggle.json file (Kaggle -> Settings -> API -> Create New Token):')
    up = files.upload()
    os.makedirs('/root/.kaggle', exist_ok=True)
    shutil.move(list(up)[0], '/root/.kaggle/kaggle.json'); os.chmod('/root/.kaggle/kaggle.json', 0o600)
!python scripts/download_data.py --out /content/MultiHazard

In [ ]:
# 4. Run all experiments (about 1-2 h). Re-run this cell after a disconnect; finished steps are skipped.
import os
os.environ['DAMGF_DATA'] = '/content/MultiHazard'
os.environ['DAMGF_OUT'] = f'{RUN}/results/run'
os.environ['DAMGF_EXPORT'] = f'{RUN}/data'
!python src/damgf_experiments.py 2>&1 | tee -a {RUN}/run_log.txt
!pip freeze | grep -iE "^(torch|torchvision|timm|transformers|scikit-learn|numpy|pandas|imagehash|fvcore)==" > {RUN}/results/run/environment.txt
!python --version >> {RUN}/results/run/environment.txt
!nvidia-smi --query-gpu=name --format=csv,noheader >> {RUN}/results/run/environment.txt

In [ ]:
# 5. Show the summary
from IPython.display import Markdown, display
display(Markdown(open(f'{RUN}/results/run/RESULTS_FOR_PAPER.md').read()))

In [ ]:
# 6. Download the files to put in the GitHub repository
import os
os.chdir(RUN)
!rm -f /content/DAMGF-Net_outputs.zip
!zip -qr /content/DAMGF-Net_outputs.zip data results -x "results/run/cache/*" "results/run/image_manifest*.csv"
from google.colab import files
files.download('/content/DAMGF-Net_outputs.zip')
print('Unzip this file into the repository folder: its data/ and results/ folders go into the repository root')